        # Scenario Bundle Tutorial 02 - Re-import a scenario bundle from a pipeline

        <!-- keep img below title and without align="left"  -->
<img src="https://raw.githubusercontent.com/OpenEnergyPlatform/organisation/refs/heads/production/logo/OpenEnergyFamily_Logo_OpenEnergyPlatform.svg" alt="OpenEnergy Platform" height="75" width="75" />

> **Verified on 2026-10-06 against OEP release v1.10.1.** The outputs
> below are a recorded run of this notebook against the live platform. If
> you see something different when you run it yourself, the API may have
> moved on since: the [API reference](https://openenergyplatform.github.io/oeplatform/oeplatform-code/web-api/api-reference/#/Scenario%20Bundles) is generated from the
> code and is always current, and the feedback issue below is the place to
> tell us.

## Learnings

A modelling pipeline that publishes its scenarios to the Open Energy
Knowledge Graph (OEKG) runs again next month, with new results. It has
kept nothing from the last run: no file with the bundle's identifier, no
copy of what it sent. This tutorial shows how such a run finds its bundle
again and makes it match its new results in a single request.

This tutorial will enable you to:

- [Find the bundle your last run left behind](#find-the-bundle-your-last-run-left-behind)
- [Declare the whole bundle in one call](#this-months-run-declare-the-whole-bundle)
- [See that a run that changes nothing writes nothing](#run-it-again-nothing-changed-nothing-written)
- [Read the runs side by side in the history](#read-the-history-the-runs-side-by-side)

It does not repeat how a bundle and its parts are created one by one, or
how the version check works. [Scenario Bundle Tutorial 01](https://openenergyplatform.github.io/academy/tutorials/05_scenario_bundles/01_scenario_bundle_api/)
does. Each of the two tutorials can be run on its own, in either order.

The API behaves according to nine rules that no endpoint signature can
state. This tutorial links them where they matter instead of repeating
them, because they can change and this page is a snapshot: read them on
[Writing scenario bundles](https://openenergyplatform.github.io/oeplatform/oeplatform-code/web-api/oekg-api/scenario-bundles/).

## Requirements

- have your OEP API token at hand. You find it in the settings tab of your
  profile page. (If you don't have a token see
  [Things you only need to do once](https://openenergyplatform.github.io/academy/tutorials/99_other/beginners_guide/#1-things-you-only-need-to-do-once).)
- your OEP user name. The tutorial names its bundle after you, see
  [setup](#setup).
- Python
- Python packages:
  - `requests` (See the packages [documentation](https://requests.readthedocs.io/en/latest/) for more help)
  - jupyter notebook, e.g. `notebook` or `jupyterlab`
- Skills:
  - use [python](https://wiki.python.org/moin/BeginnersGuide) with [jupyter notebooks](https://docs.jupyter.org/en/latest/)
  - _some_ understanding of [JSON encoded data](https://www.w3schools.com/js/js_json_datatypes.asp)
- Internet: obviously, but also make sure your network settings allow https traffic from python

## What this tutorial writes

The knowledge graph has **no sandbox**. Everything this tutorial creates is
written to the public OEKG on the Open Energy Platform and is visible to
everyone until it is deleted. The tutorial therefore deletes its bundle at
the end, and also at the start, in case an earlier run was interrupted
before it got there. If you stop half-way, run the
[first section](#find-the-bundle-your-last-run-left-behind) again to clean
up.

## Feedback

You can provide feedback on this tutorial in this [GitHub issue](https://github.com/OpenEnergyPlatform/academy/issues/TBD)

## Setup

This part of the code imports the required packages and asks for your OEP
API token and your OEP user name. Both are read from the environment
variables `OEP_API_TOKEN` and `OEP_USERNAME` if you have set them, and
asked for otherwise. Setting them keeps your token out of the notebook.

The user name decides the acronym of the bundle this tutorial works on,
`OEP-TUTORIAL-PIPELINE-<your user name>`. The acronym is how the pipeline
finds its bundle, so it has to be yours: with a shared one, two readers
would write into, and delete, each other's bundles. That is why the cell
stops if you give no user name rather than falling back to some default.
It differs from Tutorial 01's acronym, so the two tutorials never touch
each other's bundle.

The two small functions at the end print what a response says and stop
the notebook when a request did not get the answer it expected.

In [ ]:
# install required packages with: pip install requests
# import required packages
import json
from copy import deepcopy
from getpass import getpass
from os import environ

import requests as req

OEP_URL = environ.get("OEP_URL", "https://openenergyplatform.org")
BUNDLES_URL = f"{OEP_URL}/api/v0/scenario-bundles/"

token = environ.get("OEP_API_TOKEN") or getpass("Enter your OEP API token:")
username = environ.get("OEP_USERNAME") or input("Enter your OEP user name:")
if not username.strip():
    raise ValueError(
        "This tutorial needs your OEP user name: it names its bundle after "
        "you, so that it can never write into somebody else's."
    )
ACRONYM = f"OEP-TUTORIAL-PIPELINE-{username.strip()}"

# for writing, we need to add the authorization header
auth_headers = {"Authorization": "Token %s" % token}


def check(res, expected_status):
    """Print status and ETag of a response; stop if the status is unexpected."""
    print(res.status_code, res.reason, "| ETag:", res.headers.get("ETag"))
    if res.status_code != expected_status:
        raise Exception(res.text)


def show(data):
    print(json.dumps(data, indent=2, ensure_ascii=False))


print("The tutorial bundle's acronym is", ACRONYM)

## Find the bundle your last run left behind

This is the first thing every run of a pipeline does, and the one thing it
cannot do without: find the bundle it wrote last time. The pipeline kept
no identifier, but it knows its acronym, and acronyms are unique on the
platform. So it asks for the bundle carrying exactly that acronym
([rule 8](https://openenergyplatform.github.io/oeplatform/oeplatform-code/web-api/oekg-api/scenario-bundles/#rule-8-the-server-mints-the-identifier)).

Reading is public, so this needs no token. The answer is a short summary
of the bundle, and its `_meta` carries both things the next write needs:
the identifier for the URL, and the version for the `If-Match` header.

In [ ]:
def find_bundle(acronym):
    """The summary of the bundle carrying exactly this acronym, or None."""
    res = req.get(BUNDLES_URL, params={"acronym": acronym})
    check(res, 200)
    results = res.json()["results"]
    return results[0] if results else None


found = find_bundle(ACRONYM)
show(found)

A real pipeline would now go on and update what it found. In this
tutorial, a bundle found here was left behind by an earlier run of the
tutorial itself, so the tutorial deletes it and starts from the same
clean state every time. The delete needs the version from the summary and
the acronym once more as confirmation; Tutorial 01 explains both.

In [ ]:
if found:
    res = req.delete(
        f"{BUNDLES_URL}{found['_meta']['uid']}/",
        params={"confirm": ACRONYM},
        headers={**auth_headers, "If-Match": f'"{found["_meta"]["version"]}"'},
    )
    check(res, 200)
    print("Removed the bundle an earlier run of this tutorial left behind.")
else:
    print("Nothing left behind by an earlier run.")

## Scaffolding: last month's run

To re-import a bundle, there has to be one. This part stands in for your
pipeline's first run, last month, and is scaffolding rather than a
lesson: it creates the whole bundle in one request, with two scenarios and
the table the first of them used
([rule 4](https://openenergyplatform.github.io/oeplatform/oeplatform-code/web-api/oekg-api/scenario-bundles/#rule-4-a-bundle-post-accepts-nested-sub-resources-a-patch-does-not)).

`declare` is the part of a pipeline that turns one run's results into a
bundle. Everything except the abstract and the list of scenarios is the
same in every run.

In [ ]:
OEO = "https://openenergyplatform.org/ontology/oeo/"


def declare(abstract, scenarios):
    """The whole bundle, as one run of the pipeline describes it."""
    return {
        "label": "OEP Academy pipeline tutorial bundle",
        "acronym": ACRONYM,
        "abstract": abstract,
        "descriptors": [OEO + "OEO_00140133"],  # renewable energy share
        "sector_divisions": [OEO + "OEO_00010056"],  # KSG sector division
        "sectors": [OEO + "OEO_00010065"],  # KSG sector energy industry
        "technologies": [
            OEO + "OEO_00010424",  # wind power technology
            OEO + "OEO_00010428",  # photovoltaic technology
        ],
        "scenarios": scenarios,
    }


HIGH_RE = {
    "label": "A high renewables scenario",
    "acronym": "HIGH-RE",
    "scenario_types": [OEO + "OEO_00020247"],  # target driven scenario
    "years": ["2030-01-01T00:00:00Z"],
    "datasets": [{"type": "input", "ref": "table", "name": "tutorial_example_table"}],
}
LOW_RE = {
    "label": "A low renewables scenario",
    "acronym": "LOW-RE",
    "scenario_types": [OEO + "OEO_00020247"],  # target driven scenario
    "years": ["2030-01-01T00:00:00Z"],
}
REFERENCE = {
    "label": "A reference scenario",
    "acronym": "REF",
    "scenario_types": [OEO + "OEO_00020314"],  # reference scenario
    "years": ["2030-01-01T00:00:00Z"],
}

last_month = declare("Written by the OEP Academy pipeline tutorial.", [HIGH_RE, LOW_RE])

res = req.post(BUNDLES_URL, json=last_month, headers=auth_headers)
check(res, 201)
print("Created", ACRONYM, "with the scenarios")
for scenario in res.json()["scenarios"]:
    print(" ", scenario["acronym"], scenario["_meta"]["uid"])

## This month's run: declare the whole bundle

This month the study changed: the low renewables scenario was dropped, a
reference scenario was added, and the abstract says so. A pipeline does
not want to work out which requests turn last month's bundle into this
month's. It wants to state what the bundle is now, and that is what
`POST /api/v0/scenario-bundles/<uid>/replace/` takes: the complete bundle,
scenarios and dataset links included, which it makes the graph match in
one write.

**Leaving something out deletes it, at this one address only.** Everywhere
else in the API, a key you do not send is left alone. Here, a scenario,
study report or dataset link missing from the declaration is deleted, and
a field you leave out is emptied. Read
[rule 3](https://openenergyplatform.github.io/oeplatform/oeplatform-code/web-api/oekg-api/scenario-bundles/#rule-3-delete-by-omission-lives-on-replace-and-nowhere-else)
before you point a pipeline at it.

One run of the pipeline is therefore:

1. find the bundle by its acronym, which gives its identifier and the
   version for `If-Match`;
2. read the bundle, to learn the identifiers of the parts it already has;
3. give every part the declaration keeps its identifier in `_meta.uid`, so
   the replace updates it in place. A part with no identifier is created,
   and a part that is not in the declaration at all is deleted;
4. send the declaration to `replace/`.

Step 3 is the one that is easy to get wrong. Without the identifiers, the
replace would delete every scenario and dataset link and create them anew
on every run. And a scenario sent without its `datasets` would lose its
citations.

In [ ]:
def keep_identifiers(declaration, current):
    """Give each part the declaration keeps the identifier it already has."""
    scenarios = {s["acronym"]: s for s in current["scenarios"]}
    for scenario in declaration["scenarios"]:
        existing = scenarios.get(scenario["acronym"])
        if existing is None:
            continue  # a new scenario: the server mints its identifier
        scenario["_meta"] = {"uid": existing["_meta"]["uid"]}
        links = {
            (link["type"], link["ref"], link["name"]): link["_meta"]["uid"]
            for link in existing["datasets"]
        }
        for link in scenario.get("datasets", []):
            uid = links.get((link["type"], link["ref"], link["name"]))
            if uid:
                link["_meta"] = {"uid": uid}


def publish(declaration):
    """One run of the pipeline: make its bundle what the declaration says."""
    found = find_bundle(declaration["acronym"])
    uid = found["_meta"]["uid"]
    precondition = f'"{found["_meta"]["version"]}"'

    res = req.get(f"{BUNDLES_URL}{uid}/")
    check(res, 200)

    declaration = deepcopy(declaration)
    keep_identifiers(declaration, res.json())

    res = req.post(
        f"{BUNDLES_URL}{uid}/replace/",
        json=declaration,
        headers={**auth_headers, "If-Match": precondition},
    )
    check(res, 200)
    return res.json()

In [ ]:
this_month = declare(
    "Written by the OEP Academy pipeline tutorial. "
    "The low renewables scenario was dropped and a reference scenario added.",
    [HIGH_RE, REFERENCE],
)

result = publish(this_month)

print("version:", result["_meta"]["version"])
print("deleted:")
show(result["_meta"]["deleted"])
print("unlinked:")
show(result["_meta"]["unlinked"])
print("scenarios:")
for scenario in result["scenarios"]:
    links = [link["name"] for link in scenario["datasets"]]
    print(" ", scenario["acronym"], scenario["_meta"]["uid"], links)

`deleted` names the low renewables scenario (compare its identifier with
the ones the scaffolding printed), and nothing in the request named it:
leaving it out is what deleted it. The high renewables scenario kept its
identifier and its dataset link, because the declaration carried both
identifiers. The reference scenario is new, with an identifier the
server minted. `unlinked` would list a part that another bundle also uses:
that part is detached from this bundle rather than deleted.

## Run it again: nothing changed, nothing written

A pipeline that runs every night mostly finds nothing new. Declaring a
bundle exactly as it already is writes nothing at all: the version stays
where it is, nothing is deleted, and the history records nothing. So a
nightly run does not fill the bundle's history with changes it never
made.

In [ ]:
result = publish(this_month)

print("version:", result["_meta"]["version"])
print("deleted:", result["_meta"]["deleted"])
print("unlinked:", result["_meta"]["unlinked"])

## Read the history: the runs side by side

The bundle's history is public and newest first. It holds one entry for
last month's run, which created the bundle, and one for this month's
replace. The run that changed nothing has no entry. The list below
summarises each entry: the request, the resource it was about, the
versions either side of it, who made it and which fields of that resource
it set. The scenarios the replace added and deleted are in the full
entry's `changes`.

In [ ]:
# the OEO classes of what a history entry can be about
KINDS = {
    OEO + "OEO_00020227": "bundle",
    OEO + "OEO_00000365": "scenario",
    OEO + "OEO_00030029": "dataset link",
}

uid = find_bundle(ACRONYM)["_meta"]["uid"]
res = req.get(f"{BUNDLES_URL}{uid}/history/")
check(res, 200)

for entry in res.json()["results"]:
    fields = sorted({c["field"] for c in entry["changes"] or [] if c["field"]})
    print(
        entry["timestamp"][:19],
        entry["verb"].ljust(7),
        KINDS.get(entry["resource"]["type"], "other").ljust(12),
        f"version {entry['version_before']} -> {entry['version_after']}",
        "by",
        entry["actor"],
        "|",
        ", ".join(fields) or "-",
    )

## Clean up

The tutorial ends by deleting its bundle, so that it leaves nothing in the
public knowledge graph. This is the same two-step delete as at the start:
the version found by acronym in `If-Match`, and the acronym again in
`?confirm=`.

In [ ]:
found = find_bundle(ACRONYM)

res = req.delete(
    f"{BUNDLES_URL}{found['_meta']['uid']}/",
    params={"confirm": ACRONYM},
    headers={**auth_headers, "If-Match": f'"{found["_meta"]["version"]}"'},
)
check(res, 200)
kinds = [KINDS.get(node["type"], "other") for node in res.json()["deleted"]]
print("deleted:", ", ".join(kinds))

## Now do it for real

In your pipeline, give the bundle its own acronym, let `declare` build the
bundle from your results, and leave out the final delete. The very first
run finds nothing and creates the bundle, every later run replaces it:

```python
declaration = declare(abstract, scenarios)  # from this run's results
if find_bundle(declaration["acronym"]) is None:
    res = req.post(BUNDLES_URL, json=declaration, headers=auth_headers)
    check(res, 201)
else:
    publish(declaration)
```

- [Writing scenario bundles](https://openenergyplatform.github.io/oeplatform/oeplatform-code/web-api/oekg-api/scenario-bundles/): the nine rules that decide what a
  request does, with rule 3 on `replace/`.
- [API reference](https://openenergyplatform.github.io/oeplatform/oeplatform-code/web-api/api-reference/#/Scenario%20Bundles/scenario_bundles_replace_create): `replace/`
  and every other endpoint, header and status code.
- [Scenario Bundle Tutorial 01](https://openenergyplatform.github.io/academy/tutorials/05_scenario_bundles/01_scenario_bundle_api/): creating and changing a
  bundle step by step, and the version check.

## About this tutorial

:oep-logo-nfdi4energy:

- Author: Jonas Huber
- Copyright: [Reiner Lemoine Institut](https://reiner-lemoine-institut.de) (2026)
- License: [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/deed.en)
- Attribution: Reiner Lemoine Institut (2026): Scenario Bundle Tutorial 02 - Re-import a scenario bundle from a pipeline.
- Last update :oep-auto-lastupdate: